# Clean HubMonthlyUsers

Cleans a raw `HubMonthlyUsers_yyyy-mm-dd.csv` export.

**Deviations from a literal reading of the notes, carried over from the script:**
- *"Remove completely blank rows"* is evaluated against the `ls_cols` fields only, not the full raw column set. Two rows are blank across every `ls_cols` field but still have populated pipeline-metadata columns (`DataSource`, `PipelineRunID`, ...) that get dropped later — checking against the raw columns would miss them and crash the date conversion on an empty string.
- Rows where `MonthDate` (post-rename) is blank are also dropped explicitly, as a safety net matching the three daily notebooks — on the current file this is a no-op (the 2 blank-`MonthDate` rows are already caught by the completely-blank check above), but it guards against a future file where `Date` is blank while other fields aren't, which would otherwise crash the `strftime` conversion.
- This notebook deliberately does **not** read with `engine="python", escapechar="\\"` (unlike the three daily notebooks). The raw data contains a real company name, `TBWA\RAAD` (a genuine single backslash, not a CSV escape artifact — this file has no HTML content that would need escaped quotes), and `escapechar="\\"` would silently strip that backslash to `TBWARAAD`. A plain read is correct here.
- The file is read and written with `encoding="utf-8"` explicitly, since this environment's OS default encoding (`cp1252`) would corrupt accented characters (e.g. "Côte d'Ivoire") otherwise.


## Imports

In [1]:
import csv
import re
from pathlib import Path

import pandas as pd


## Schema constants

- `LS_COLS` — the final column set and order for the cleaned output.
- `LS_STRING_COLS` — the free-text columns that get whitespace-trimmed.
- `LS_INT_COLS` — the numeric columns that get cast to integer type.
- `FILENAME_RE` — extracts the year/month from the input filename, used both to build `month_tag` and to auto-detect the input file.
- `GROUP_COLS`/`SUM_COLS` — used to collapse duplicate rows (see "Collapse duplicate rows" below): every `LS_COLS` field except `LS_INT_COLS` is a group-by key, and `LS_INT_COLS` is what gets summed.


In [2]:
LS_COLS = [
    "MonthDate", "Month", "CompanyCode", "CompanyName", "Country",
    "HomeCountry", "HomeCountryCode", "Operation",
    "NewUsers", "Users", "Sessions", "Hits",
]
LS_STRING_COLS = [
    "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode", "Operation",
]
LS_INT_COLS = ["NewUsers", "Users", "Sessions", "Hits"]

FILENAME_RE = re.compile(r"^HubMonthlyUsers_(\d{4})-(\d{2})-\d{2}\.csv$")

# Duplicate rows are collapsed by grouping on every LS_COLS field except the summed
# measures (NewUsers/Users/Sessions/Hits) and summing those.
GROUP_COLS = [c for c in LS_COLS if c not in LS_INT_COLS]
SUM_COLS = LS_INT_COLS


## Locate the input file

`find_default_input` looks for a single `HubMonthlyUsers_yyyy-mm-dd.csv` file in a given directory and returns it automatically. If none or several are found, it raises rather than silently guessing which one to use.


In [3]:
def find_default_input(directory: Path) -> Path:
    matches = sorted(p for p in directory.glob("HubMonthlyUsers_*.csv") if FILENAME_RE.match(p.name))
    if not matches:
        raise FileNotFoundError(f"No HubMonthlyUsers_yyyy-mm-dd.csv file found in {directory}")
    if len(matches) > 1:
        raise ValueError(
            f"Multiple candidate input files found in {directory}: "
            f"{[m.name for m in matches]}. Pass one explicitly."
        )
    return matches[0]


## Derive `month_tag` from the filename

The output name has the format of `HubMonthlyUsers_<month_tag>_cleaned.csv`, where `month_tag` is `yyyymm` for the month *before* the input filename's `yyyy-mm-dd` date suffix (the export date's month minus one), since the export reports on the prior month's usage.

In [4]:
def month_tag_from_filename(path: Path) -> str:
    match = FILENAME_RE.match(path.name)
    if not match:
        raise ValueError(f"Filename '{path.name}' does not match expected pattern HubMonthlyUsers_yyyy-mm-dd.csv")
    year, month = (int(g) for g in match.groups())
    # month_tag refers to the prior month's data, not the export date's month.
    year, month = (year - 1, 12) if month == 1 else (year, month - 1)
    return f"{year}{month:02d}"


## Cleaning logic

The core transformation, in the order implemented:

1. Rename `Date` → `MonthDate`.
2. Drop rows that are blank across every `ls_cols` field (see the note at the top of this notebook on why this checks `ls_cols` rather than the raw columns).
3. Drop rows where `MonthDate` is blank.
4. Reformat `MonthDate` to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds) and `Month` to `%Y %m`.
5. Reorder/drop columns to match `ls_cols`.
6. Trim surrounding whitespace on the string columns.
7. Cast `NewUsers`, `Users`, `Sessions`, `Hits` to integer type.


In [5]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={"Date": "MonthDate"})

    # Same reasoning as the daily notebooks: judge "completely blank" against the
    # ls_cols fields only, since raw pipeline-metadata columns (DataSource, PipelineRunID,
    # FileName, ...) are dropped later and would otherwise mask genuinely blank rows.
    present_ls_cols = [c for c in LS_COLS if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    df = df.loc[~is_blank].copy()

    # Only MonthDate being blank drops a row, matching the same safety net the three
    # daily notebooks already have (guards against the strftime call below crashing
    # on an empty string).
    missing_key = df["MonthDate"].str.strip() == ""
    df = df.loc[~missing_key].copy()

    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["MonthDate"] = pd.to_datetime(df["MonthDate"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]
    df["Month"] = pd.to_datetime(df["Month"], format="%Y-%m").dt.strftime("%Y %m")

    df = df[[c for c in LS_COLS if c in df.columns]]

    for col in LS_STRING_COLS:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS:
        df[col] = df[col].astype(int)

    return df


## Collapse duplicate rows

Duplicates are not allowed in the final cleaned dataset. Rows that share every `GROUP_COLS` value are collapsed into one row, summing `NewUsers`/`Users`/`Sessions`/`Hits` as integers.


In [6]:
def collapse_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in SUM_COLS:
        df[col] = df[col].astype(int)
    df = df.groupby(GROUP_COLS, as_index=False)[SUM_COLS].sum()
    return df[LS_COLS]


## Configure the input file

Leave `INPUT_FILE` as `None` to auto-detect the single raw file in this notebook's `input/` folder, or set it to an explicit path to override (equivalent to the script's optional CLI argument).

In [7]:
NOTEBOOK_DIR = Path.cwd()
INPUT_FILE = None  # e.g. "input/HubMonthlyUsers_2026-08-02.csv"

input_path = Path(INPUT_FILE).resolve() if INPUT_FILE else find_default_input(NOTEBOOK_DIR / "input")
month_tag = month_tag_from_filename(input_path)
input_path, month_tag


(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/HUB/input/HubMonthlyUsers_2026-08-02.csv'),
 '202607')

## Read the raw CSV

Read everything as strings (`dtype=str`, `keep_default_na=False`) so numeric-looking codes, leading zeros, and literal text like `"null"` in `HomeCountryCode` pass through unchanged instead of being coerced or turned into `NaN`. `encoding="utf-8"` matches the source file and avoids corrupting accented text.


In [8]:
df_raw = pd.read_csv(input_path, sep=";", dtype=str, keep_default_na=False, encoding="utf-8")
df_raw.shape


(6183, 20)

## Apply the cleaning steps

In [9]:
df_cleaned = clean(df_raw)
df_cleaned.head()


,MonthDate,Month,CompanyCode,CompanyName,Country,HomeCountry,HomeCountryCode,Operation,NewUsers,Users,Sessions,Hits
0,2026-07-01 00:00:00.000,2026 07,GIAHUB,Gemological Institute of America,Botswana,Botswana,BW,ICAS Botswana,1,1,1,4
1,2026-07-01 00:00:00.000,2026 07,LUCARAHUB,Lucara Botswana,Botswana,Botswana,BW,ICAS Botswana,6,8,8,41
2,2026-07-01 00:00:00.000,2026 07,PULAHUB,Pula Medical Aid Fund,Botswana,Botswana,BW,ICAS Botswana,6,8,11,123
3,2026-07-01 00:00:00.000,2026 07,ICASBWTEST,ICAS Botswana (Test Preview),Botswana,Botswana,BW,ICAS Botswana,2,2,3,27
4,2026-07-01 00:00:00.000,2026 07,ORANGEHUB,Orange Botswana,Botswana,Botswana,BW,ICAS Botswana,9,9,19,236


## Collapse duplicate rows before saving

`rows_before_dedup` is kept so the report below can still report "blank rows dropped" against the pre-dedup count, separately from rows collapsed for being duplicates.


In [10]:
rows_before_dedup = len(df_cleaned)
df_cleaned = collapse_duplicates(df_cleaned)
duplicates_collapsed = rows_before_dedup - len(df_cleaned)

print(f"Collapsed {duplicates_collapsed} duplicate rows -> {len(df_cleaned)} rows remaining")


Collapsed 10 duplicate rows -> 6171 rows remaining


## Save the cleaned dataset

Written as `;`-delimited UTF-8 with minimal quoting, matching the input file's own semicolon delimiter as required by the notes. Saved to this notebook's `output/` folder.

In [11]:
output_dir = NOTEBOOK_DIR / "output"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"HubMonthlyUsers_{month_tag}_cleaned.csv"
df_cleaned.to_csv(output_path, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned)} rows -> {output_path}")


Cleaned 6171 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\output\HubMonthlyUsers_202607_cleaned.csv


## Write summary report

Writes a plain-text report answering: which input file was read, the raw and cleaned row counts, how many duplicate rows exist in each of the raw and cleaned dataframes, the derived `month_tag`, how many blank rows were dropped, and the output CSV's name, followed (after three blank lines) by `df_cleaned.describe()`. Saved to this notebook's `reports/` folder as `HubMonthlyUsers_<month_tag>_report.txt`.

Duplicate counts use pandas' default `duplicated()` (`keep="first"`), i.e. the number of rows that are repeats of an earlier row — how many rows would go away if the dataframe were deduplicated.

In [12]:
report_lines = [
    f"Input file: {input_path.name}",
    f"Raw row count: {len(df_raw)}",
    f"Raw duplicate rows: {int(df_raw.duplicated().sum())}",
    "=======================================================================",
    f"Month tag: {month_tag}",
    f"Blank rows dropped: {len(df_raw) - rows_before_dedup}",
    f"Duplicate rows collapsed: {duplicates_collapsed}",
    "=======================================================================",
    f"Cleaned row count: {len(df_cleaned)}",
    f"Cleaned duplicate rows: {int(df_cleaned.duplicated().sum())}",
    f"Output file: {output_path.name}",
]
report_text = "\n".join(report_lines) + "\n"
report_text += "\n\n\n" + df_cleaned.describe().to_string() + "\n"

reports_dir = NOTEBOOK_DIR / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
report_path = reports_dir / f"HubMonthlyUsers_{month_tag}_report.txt"
report_path.write_text(report_text, encoding="utf-8")

print(report_text)
print(f"Report written -> {report_path}")


Input file: HubMonthlyUsers_2026-08-02.csv
Raw row count: 6183
Raw duplicate rows: 0
Month tag: 202607
Blank rows dropped: 2
Duplicate rows collapsed: 10
Cleaned row count: 6171
Cleaned duplicate rows: 0
Output file: HubMonthlyUsers_202607_cleaned.csv



          NewUsers        Users      Sessions          Hits
count  6171.000000  6171.000000   6171.000000   6171.000000
mean      7.352941     8.898234     24.237401     63.422946
std     103.521488   122.471859    273.691851    530.360765
min       0.000000     1.000000      1.000000      1.000000
25%       1.000000     1.000000      1.000000      2.000000
50%       1.000000     1.000000      2.000000      6.000000
75%       2.000000     3.000000      5.000000     22.000000
max    6520.000000  7890.000000  10324.000000  23512.000000

Report written -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\reports\HubMonthlyUsers_202607_report.txt


In [13]:
df_cleaned.describe()

,NewUsers,Users,Sessions,Hits
count,6171.000000,6171.000000,6171.000000,6171.000000
mean,7.352941,8.898234,24.237401,63.422946
std,103.521488,122.471859,273.691851,530.360765
min,0.000000,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000,2.000000
50%,1.000000,1.000000,2.000000,6.000000
75%,2.000000,3.000000,5.000000,22.000000
max,6520.000000,7890.000000,10324.000000,23512.000000
